### Procesamiento de Lenguaje Natural 
# Desafio 2

In [2]:
import os
import platform

if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
import pandas as pd
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# Cargar el corpus (mismo dataset, otro artista)
df = pd.read_csv('songs_dataset/eminem.txt', sep='/n', header=None, engine='python')
print("Cantidad de documentos:", df.shape[0])

# Tokenizar cada línea en una lista de palabras
sentence_tokens = []
for _, row in df.iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

sentence_tokens[:2]  # vistazo rápido

Cantidad de documentos: 6812


[['look',
  'i',
  'was',
  'gonna',
  'go',
  'easy',
  'on',
  'you',
  'and',
  'not',
  'to',
  'hurt',
  'your',
  'feelings'],
 ['but', "i'm", 'only', 'going', 'to', 'get', 'this', 'one', 'chance']]

In [4]:
import pandas as pd
from tensorflow.keras.preprocessing.text import text_to_word_sequence
df = pd.read_csv('songs_dataset/eminem.txt', sep='/n', header=None, engine='python')

In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 6812


In [6]:
sentence_tokens = []
for _, row in df.iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))
sentence_tokens[:2]

[['look',
  'i',
  'was',
  'gonna',
  'go',
  'easy',
  'on',
  'you',
  'and',
  'not',
  'to',
  'hurt',
  'your',
  'feelings'],
 ['but', "i'm", 'only', 'going', 'to', 'get', 'this', 'one', 'chance']]

In [7]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

# Callback para ver el loss en cada época (opcional pero útil para ver si converge)
class callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print(f'Loss after epoch {self.epoch}: {loss}')
        else:
            print(f'Loss after epoch {self.epoch}: {loss - self.loss_previous_step}')
        self.epoch += 1
        self.loss_previous_step = loss

# Instanciamos el modelo (todavía sin entrenar)
w2v_model = Word2Vec(
    min_count=5,
    window=3,
    vector_size=200,
    negative=15,
    workers=1,
    sg=1
)

# Construye el vocabulario a partir de nuestras oraciones
w2v_model.build_vocab(sentence_tokens)
print("Documentos en el corpus:", w2v_model.corpus_count)
print("Palabras distintas en el vocabulario:", len(w2v_model.wv.index_to_key))

Documentos en el corpus: 6812
Palabras distintas en el vocabulario: 1347


In [9]:
w2v_model.train(
    sentence_tokens,
    total_examples=w2v_model.corpus_count,
    epochs=20,
    compute_loss=True,
    callbacks=[callback()]
)

Loss after epoch 0: 552502.125
Loss after epoch 1: 423219.625
Loss after epoch 2: 399855.875
Loss after epoch 3: 386707.625
Loss after epoch 4: 370746.5
Loss after epoch 5: 314481.5
Loss after epoch 6: 307463.5
Loss after epoch 7: 304598.5
Loss after epoch 8: 300046.5
Loss after epoch 9: 294587.0
Loss after epoch 10: 292555.25
Loss after epoch 11: 286955.5
Loss after epoch 12: 262533.5
Loss after epoch 13: 263743.0
Loss after epoch 14: 262185.0
Loss after epoch 15: 260263.0
Loss after epoch 16: 259005.0
Loss after epoch 17: 258385.0
Loss after epoch 18: 257041.5
Loss after epoch 19: 254502.5


(786250, 1296760)

El loss disminuye un poco mas de la mitad, con una gran caida al inciio y luego se estabiliza por lo q no hacen falta mas epocas.

In [10]:
# Palabras que mas se relacionan con...
for word in ["fuck", "life", "pain", "love"]:
    print(f"\n--- {word} ---")
    try:
        for similar_word, score in w2v_model.wv.most_similar(word, topn=5):
            print(f"{similar_word}: {score:.3f}")
    except KeyError:
        print(f"'{word}' no está en el vocabulario (apareció menos de min_count veces)")


--- fuck ---
fack: 0.652
clock: 0.600
papa: 0.579
elvis: 0.577
doc: 0.574

--- life ---
lane: 0.734
livin': 0.720
twenty: 0.700
five: 0.689
fast: 0.676

--- pain ---
smile: 0.761
tired: 0.742
kind: 0.728
rush: 0.716
return: 0.694

--- love ---
showed: 0.703
ladies: 0.703
lie: 0.691
hurts: 0.631
crying: 0.620


salvo en la 1ra palabra que no encuentro mucha coherencia, las demas palabras parecieran tener relaciones semanticas claras con sus vectores mas cercanos lo cual es bueno. El embedding parece capturar bien las relaciones semanticas entre las palabras

In [11]:
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions=2):
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)
    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)
    return vectors, labels

vecs, labels = reduce_dimensions(w2v_model)

In [12]:
import plotly.express as px

MAX_WORDS = 150  # limitamos para que el gráfico no sea ilegible

fig = px.scatter(
    x=vecs[:MAX_WORDS, 0],
    y=vecs[:MAX_WORDS, 1],
    text=labels[:MAX_WORDS]
)
fig.update_traces(textposition='top center')
fig.show()

Puedo observar que se amontonan palabras en algunas zonas, y son palabras q si tienen relacion entre ellas y seria normal encontrarlas juntas en una oración como "get, you, to, try" o "need, no, more" y esta bien pq word2vec aprende la similitud en los contextos en los q son usadas las palabras. Tambien veo palabras aisladas q se me ocurre q son palabras de uso muy particular

In [13]:
# Palabras que MENOS se relacionan con...
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('hands', -0.016827229410409927),
 ('his', -0.026485832408070564),
 ('box', -0.04020070657134056),
 ('slap', -0.06390079110860825),
 ('an', -0.07588297128677368),
 ('fire', -0.08799269050359726),
 ('enough', -0.08891212940216064),
 ('from', -0.10096384584903717),
 ('whole', -0.10165105015039444),
 ('dynamite', -0.10532255470752716)]

Ahora en vez de buscar vectores cercanos, busca los más opuestos en el espacio vectorial, palabras que casi nunca comparten contexto con la que elegi